# KJV RAG: retrieval walkthrough

This notebook builds the **retrieval** half of a RAG (retrieval-augmented generation) system over the King James Bible, one step at a time.

The goal: given a question like *"Jonah is swallowed by a big fish"*, find the Bible passages that answer it, so an LLM can later write an answer from them and cite the verses.

| Step | What happens | Code |
|---|---|---|
| 1. Load | Read the text out of the PDF | `app/chunking.py` |
| 2. Clean and parse | Turn messy page text into 31,102 clean verses | `app/chunking.py` |
| 3. Chunk | Group verses into passages (parents) and small search windows (children) | `app/chunking.py` |
| 4. Embed | Turn every child into a vector with two different models | `app/embeddings.py` |
| 5. Store | Put the vectors in FAISS and in Chroma | `app/dense_retriever.py` |
| 6. Pick a model | Ask 54 test questions and score which model finds the right passage | `app/evaluation.py` |
| 7. Pick a technique | Score ten retrieval techniques on the same questions and choose the best | `app/techniques.py` |
| 8. Rerank | Re-order the best technique's results with a bi-encoder or a cross-encoder, and choose the best | `app/reranker.py` |

The notebook only *calls* the code in `app/`, so the same steps also run from the command line:

```
python -m app.main ingest      # steps 1-3
python -m app.main embed       # steps 4-5
python -m app.main evaluate    # step 6
python -m app.main compare     # step 7
python -m app.main rerank      # step 8
python -m app.main search "Jonah is swallowed by a big fish"   # uses the final search: Hybrid + bge reranking
```

In [1]:
#quieter output: hide model-loading progress bars and Hugging Face notices
import os
import warnings

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"
warnings.filterwarnings("ignore", message="IProgress not found")

import transformers

transformers.logging.set_verbosity_error()
transformers.logging.disable_progress_bar()

from app import chunking, embeddings, evaluation
from app.dense_retriever import ChromaRetriever, FaissRetriever, parents_for

PDF_PATH = "data/KJVfile.pdf"
DATA_DIR = "data"


## 1. Load the PDF

The PDF has 1,517 pages. Page 1 is a table of contents; the Bible text starts on page 2.

Here is what the raw extracted text of page 2 looks like. It is readable, but not ready to use: verse numbers sit on their own lines, headings and cross-references are mixed into the text, and lines break in the middle of sentences.

In [2]:
import pymupdf

with pymupdf.open(PDF_PATH) as document:
    print(f"{len(document)} pages\n")
    print(document[1].get_text()[:600])

1517 pages

Genesis 1 
KJV  [Online] 
The Creation 
(John 1:1–5; Hebrews 11:1–3) 
1
 In the beginning God created the heaven and the earth. 2
 And the earth was without form, and void; 
and darkness [was] upon the face of the deep. And the Spirit of God moved upon the face of the 
waters.  
The First Day 
3
 And God said, Let there be light: and there was light. 4
 And God saw the light, that [it was] good: and 
God divided the light from the darkness. 5
 And God called the light Day, and the darkness he called 
Night. And the evening and the morning were the first day.  
The Second Day 
6
 And God said, 


## 2. Clean and parse into verses

Instead of generic text cleaning, we use the fact that this PDF has a **regular layout**:

| Pattern in the text | What it is | What we do |
|---|---|---|
| `Genesis 1` then `KJV  [Online]` | Chapter header | Start a new chapter |
| `1`, a newline, then a narrow no-break space | Verse number | Start a new verse |
| `The Creation`, `The First Day` | Section heading | Keep it as **metadata**, remove it from the verse text |
| `(John 1:1–5; Hebrews 11:1–3)` | Cross-reference | Remove it |
| A verse that continues on the next page | Page break | Join all pages **before** parsing |

The result is one clean record per verse: book, chapter, verse number, section heading and text.

In [3]:
full_text = chunking.read_pdf_text(PDF_PATH)
verses = chunking.parse_verses(full_text)

print(f"{len(verses)} verses")
verses[:3]

31102 verses


[{'book': 'Genesis',
  'chapter': 1,
  'verse': 1,
  'heading': 'The Creation',
  'text': 'In the beginning God created the heaven and the earth.'},
 {'book': 'Genesis',
  'chapter': 1,
  'verse': 2,
  'heading': 'The Creation',
  'text': 'And the earth was without form, and void; and darkness [was] upon the face of the deep. And the Spirit of God moved upon the face of the waters.'},
 {'book': 'Genesis',
  'chapter': 1,
  'verse': 3,
  'heading': 'The First Day',
  'text': 'And God said, Let there be light: and there was light.'}]

### Check the result

The KJV has exactly **66 books, 1,189 chapters and 31,102 verses**, so we can check the parsing against known numbers.

We also list verses that don't end in punctuation, because a heading accidentally glued onto a verse would show up there. The few that appear are real KJV verses that end without punctuation.

In [4]:
assert len({v["book"] for v in verses}) == 66
assert len({(v["book"], v["chapter"]) for v in verses}) == 1189
assert len(verses) == 31102
print("66 books, 1,189 chapters, 31,102 verses: all correct\n")

no_punctuation = [v for v in verses if v["text"][-1] not in ".,;:?!)]'" and not v["text"].endswith(chr(0x2019))]
for v in no_punctuation:
    print(f'{v["book"]} {v["chapter"]}:{v["verse"]}  ...{v["text"][-50:]}')

66 books, 1,189 chapters, 31,102 verses: all correct

Genesis 23:17  ...re] in all the borders round about, were made sure
1 Chronicles 21:11  ...nd said unto him, Thus saith the LORD, Choose thee
2 Chronicles 30:18  ...d for them, saying, The good LORD pardon every one
Psalms 96:12  ...rein: then shall all the trees of the wood rejoice
Psalms 98:8  ...ap [their] hands: let the hills be joyful together
Romans 11:7  ...ection hath obtained it, and the rest were blinded
Colossians 1:21  ...] mind by wicked works, yet now hath he reconciled


## 3. Chunk: parent-child

A single verse is often too short to carry meaning on its own, and a whole chapter is too long to search precisely. **Parent-child chunking** uses two sizes at once:

| Level | Chunking strategy | Size | Used for |
|---|---|---|---|
| **Parent** | Document-structure-based: split at book, chapter and section-heading boundaries | Up to 10 verses | What the LLM **reads** |
| **Child** | Fixed-size with overlap, inside each parent | 3 verses, overlapping by 1 | What we **search** |

At question time we search the small children (precise), then hand the LLM their parents (full context).

The overlap is only on the children: each child belongs to exactly one parent, so the LLM never gets the same verses twice.

In [5]:
parents, parent_verses = chunking.build_parents(verses)
children = chunking.build_children(parents, parent_verses)
parents_by_id = {p["id"]: p for p in parents}

print(f"{len(parents)} parents, {len(children)} children\n")

#one parent and its children, to see the overlap (each child shares one verse with the next)
parent = parents_by_id["Genesis-1-14-19"]
print("PARENT:", parent["reference"], "-", parent["heading"])
for child in children:
    if child["parent_id"] == parent["id"]:
        print("  child:", child["reference"])

4723 parents, 14618 children



PARENT: Genesis 1:14-19 - The Fourth Day
  child: Genesis 1:14-16
  child: Genesis 1:16-18
  child: Genesis 1:18-19


In [6]:
#save every level, so later steps don't need to re-read the PDF
chunking.write_jsonl(f"{DATA_DIR}/verses.jsonl", verses)
chunking.write_jsonl(f"{DATA_DIR}/parents.jsonl", parents)
chunking.write_jsonl(f"{DATA_DIR}/children.jsonl", children)

## 4. Embed

### What is an embedding?

An **embedding model** turns a piece of text into a list of numbers (a *vector*). Texts with similar meaning get vectors that point in a similar direction, so *"Jonah is swallowed by a big fish"* lands close to *"Now the LORD had prepared a great fish to swallow up Jonah"*, even though they share few words.

We embed every **child**. At question time we embed the question the same way and look for the closest children.

### The two models we compare

| | `all-MiniLM-L6-v2` | `all-mpnet-base-v2` |
|---|---|---|
| Short name here | `minilm` | `mpnet` |
| Vector size | 384 numbers | 768 numbers |
| Model size | 6 layers, ~22 million parameters | 12 layers, ~110 million parameters |
| Time to embed all 14,618 children on this CPU | **about 6 minutes** | **about 45 minutes** |
| Longest text it reads | 256 tokens (1 child gets cut off) | 384 tokens (none cut off) |

Embedding is a **one-time cost**: the vectors are saved to `data/embeddings/<model>.npy`, and `load_or_embed` reuses them on the next run. Embedding a single question takes milliseconds with either model.

Every vector is **normalized** (scaled to length 1), so "closest" simply means the highest *cosine similarity*.

In [7]:
child_texts = [c["text"] for c in children]

vectors = {}
for name in embeddings.MODELS:
    vectors[name] = embeddings.load_or_embed(name, child_texts, DATA_DIR)  # row i is the vector for children[i]
    print(f"{name}: {vectors[name].shape[0]} vectors of {vectors[name].shape[1]} numbers")

minilm: 14618 vectors of 384 numbers
mpnet: 14618 vectors of 768 numbers


## 5. Store the vectors: FAISS and Chroma

Both stores get **the same vectors**, so any difference in results would come from the store, not the model.

| | FAISS | Chroma |
|---|---|---|
| What it is | A fast search index, nothing more | A database: vectors plus each child's text and metadata |
| How it searches | Exact: compares the question with every child | Approximate (HNSW): almost always the same answer, much faster on huge collections |
| Finding the parent | We look up the child ourselves | `parent_id` is stored next to each vector |
| Saved to | `data/faiss/<model>.index` | `data/chroma/` |

A search does three things: embed the question with the same model, find the closest children, then return their **parents** with `parents_for()`.

In [8]:
import chromadb

chroma = chromadb.PersistentClient(path=f"{DATA_DIR}/chroma")

retrievers = {}
for name in embeddings.MODELS:
    encode = embeddings.encoder(name)
    retrievers[("FAISS", name)] = FaissRetriever.build(vectors[name], children, encode)
    retrievers[("Chroma", name)] = ChromaRetriever.build(chroma, f"kjv_children_{name}", vectors[name], children, encode)


def search(store, model, question, k=3):
    return parents_for(retrievers[(store, model)].search(question), parents_by_id, k=k)


for passage in search("FAISS", "mpnet", "Jonah is swallowed by a big fish"):
    print(passage["text"][:200], "...\n")

Jonah 1:11-17 (Jonah Cast into the Sea)
11 Then said they unto him, What shall we do unto thee, that the sea may be calm unto us? for the sea wrought, and was tempestuous. 12 And he said unto them, Ta ...

Jonah 2:1-10 (Jonah’s Prayer)
1 Then Jonah prayed unto the LORD his God out of the fish’s belly, 2 And said, I cried by reason of mine affliction unto the LORD, and he heard me; out of the belly of he ...

Jonah 1:4-10 (The Great Storm)
4 But the LORD sent out a great wind into the sea, and there was a mighty tempest in the sea, so that the ship was like to be broken. 5 Then the mariners were afraid, an ...



## 6. Which model works best?

### The test questions

To compare fairly, every model and store gets the same **54 test questions** from `data/test_questions.json`. For each question we already know which verse(s) answer it, and every answer was checked against the parsed text.

The questions come in four types, because real users ask in different ways:

| Type | How many | Example |
|---|---|---|
| **quote** | 16 | *"For God so loved the world"* |
| **paraphrase** | 20 | *"Jonah is swallowed by a big fish"* (the KJV says "great fish") |
| **names a book** | 12 | *"In Psalms, God is our refuge and strength"* |
| **topic** | 6 | *"What does the Bible say about forgiving others?"* |

### The scores

Each search returns a ranked list of passages. A passage is **relevant** if it contains at least one answer verse. We look at the **top 5**, the passages an LLM would read, with four metrics:

| Metric | The question it answers | How it's worked out | Best possible here |
|---|---|---|---|
| **Precision@5** | How much of the top 5 is useful? | relevant passages in the top 5 ÷ 5 | **0.337** (see below) |
| **Recall@5** | How much of the answer did we find? | answer verses found in the top 5 ÷ all answer verses | 1.0 |
| **MRR** (mean reciprocal rank) | How soon does the first right passage appear? | 1st place scores 1, 2nd scores 1/2, 3rd scores 1/3, a miss scores 0; then average | 1.0 |
| **nDCG@5** | Are *all* the right passages near the top? | every relevant passage counts, but less the lower it ranks (1 / log2(rank + 1)); divided by the score of the perfect order | 1.0 |

**Why precision@5 can't reach 1.0:** 37 of the 54 questions have only **one** relevant passage in the whole Bible (e.g. *"Jesus wept"*). Even a perfect search can then fill only 1 of its 5 places, scoring 0.2. Averaged over all questions, the best possible precision@5 is 0.337.

Two plain counts make the tables easier to read: **correct at #1** (questions where a relevant passage came first) and **correct in top 5**.

In [9]:
import pandas as pd

questions = evaluation.load_test_questions(f"{DATA_DIR}/test_questions.json", parents)
print("best possible precision@5:", round(sum(min(5, q["relevant_passages"]) / 5 for q in questions) / len(questions), 3))

rows, details = [], {}
for (store, name), retriever in retrievers.items():
    scores, details[(store, name)] = evaluation.evaluate(lambda q: parents_for(retriever.search(q, 50), parents_by_id, k=10), questions)
    rows.append({"store": store, "model": name, **scores})

pd.DataFrame(rows)

best possible precision@5: 0.337


,store,model,precision@5,recall@5,MRR,nDCG@5,correct at #1,correct in top 5
0,FAISS,minilm,0.263,0.821,0.741,0.729,33,49
1,Chroma,minilm,0.263,0.821,0.741,0.730,33,49
2,FAISS,mpnet,0.256,0.840,0.784,0.760,38,50
3,Chroma,mpnet,0.252,0.822,0.779,0.752,38,49


### Question by question

The position (1 = first result) at which each model found a right passage, using FAISS:

In [10]:
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 70)

pd.DataFrame({
    "type": [q["type"] for q in questions],
    "question": [q["question"] for q in questions],
    "answer": [", ".join(q["answers"]) for q in questions],
    **{name: [d["rank"] or "missed" for d in details[("FAISS", name)]] for name in embeddings.MODELS},
})

,type,question,answer,minilm,mpnet
0,quote,Am I my brother's keeper?,Genesis 4:9,3,4
1,quote,For God so loved the world,John 3:16,missed,7
2,quote,The Lord is my shepherd; I shall not want,Psalms 23:1,1,1
3,quote,Thou shalt not kill,"Exodus 20:13, Deuteronomy 5:17",1,2
4,quote,Blessed are the poor in spirit,"Matthew 5:3, Luke 6:20",1,1
5,quote,Jesus wept,John 11:35,4,1
6,quote,In the beginning was the Word,John 1:1,1,1
7,quote,"I am the way, the truth, and the life",John 14:6,1,1
8,quote,The wages of sin is death,Romans 6:23,1,1
9,quote,Let there be light,Genesis 1:3,missed,missed


### The winner: `all-mpnet-base-v2`

| Model | Correct at #1 | Correct in top 5 | MRR | Recall@5 | Time to embed |
|---|---|---|---|---|---|
| `minilm` | 33 of 54 | 49 of 54 | 0.741 | 0.821 | ~6 min |
| **`mpnet`** | **38 of 54** | **50 of 54** | **0.784** | **0.840** | ~45 min |

- **mpnet is better on every metric and every question type**, and most clearly on paraphrases (MRR 0.877 vs 0.810) and topics (0.625 vs 0.556).
- **FAISS and Chroma score almost the same** for each model (at most one question differs). That confirms both stores hold the same vectors and are set up correctly. The tiny gap comes from Chroma's *approximate* search, which occasionally misses a child that FAISS's exact search finds.
- **mpnet costs more**: about 7x longer to embed on this CPU and twice the storage. Because embedding is a one-time cost and a question still takes milliseconds, that cost is worth paying.

**Decision: use `all-mpnet-base-v2`** for every dense search from here on.

### A weakness of dense search

Both models are weak on **exact quotes** (MRR about 0.71). For example, *"For God so loved the world"* doesn't bring up John 3:16 first: dense models match *meaning*, not exact words, so they also return other passages about God, love and the world (e.g. 1 John 2:15, *"Love not the world"*). Matching exact words is what **keyword (sparse) search** is good at, which leads to the next step.

## 7. Which retrieval technique works best?

A model decides *how text becomes vectors*. A **retrieval technique** decides *how we search*: with vectors, with words, with both, with filters, or with several versions of the question. We compare ten.

### The ten techniques

| # | Technique | The idea in one line | Code |
|---|---|---|---|
| 1 | **Dense vector** | Embed each whole passage (parent); find the passages whose meaning is closest | `dense_retriever.py` |
| 2 | **Sparse (SPLADE)** | A neural model gives each passage a list of weighted keywords, including related words that aren't in the text | `sparse_retriever.py` |
| 3 | **TF-IDF** | Score passages by the words they share with the question; rare words count more | `sparse_retriever.py` |
| 4 | **BM25** | Like TF-IDF, but repeating a word has diminishing returns and long passages aren't favoured | `sparse_retriever.py` |
| 5 | **Hybrid** | Run parent-child dense search *and* BM25, then merge the two rankings | `hybrid_retriever.py` |
| 6 | **Metadata filtering** | If the question names a book or testament (*"In Psalms, …"*), only search there | `metadata_filter.py` |
| 7 | **MMR** (Maximum Marginal Relevance) | Re-rank so the results are relevant *and* different from each other, not five parts of the same story | `mmr.py` |
| 8 | **Parent-child** | Search the small 3-verse children, return their parent passages (what we did in step 6) | `dense_retriever.py` |
| 9 | **Multi-query** | Search with three versions of the question: as asked, in KJV wording (*Holy Spirit* → *Holy Ghost*), and keywords only; then merge | `multi_query.py` |
| 10 | **Contextual** | Embed each child with a line saying where it sits: *"From the book of Jonah (Old Testament, Minor Prophets), chapter 1 of 4. Section: …"* | `contextual.py` |

### Keeping the comparison fair

- **Every technique returns the same thing**: ranked parent passages. They are all scored with the same questions and the same scores.
- **Every dense part uses mpnet**, the winner of step 6.
- **Merging rankings** (Hybrid, Multi-query) uses *reciprocal rank fusion*. Each list gives a passage `1 / (60 + rank)` points, and passages are re-ranked by their total. Only ranks are used, so a cosine similarity and a BM25 score never have to be compared directly.
- **No LLM is used.** Multi-query normally asks an LLM to reword the question, and contextual retrieval normally asks an LLM to write each chunk's context. Here both use rules instead: a word list for multi-query, and the book, chapter and section headings for contextual.

The extra vectors these techniques need (mpnet vectors of the parents, SPLADE vectors and contextual children) take about 3 hours on a CPU. They were built once by `python -m app.main embed` and are loaded from `data/embeddings/` here.

In [11]:
from app.main import compare_techniques
from app.techniques import DESCRIPTIONS, build_techniques

techniques = build_techniques(DATA_DIR)
overall, by_type = compare_techniques(techniques, questions)
overall

,technique,precision@5,recall@5,MRR,nDCG@5,correct at #1,correct in top 5
1,Hybrid,0.296,0.933,0.874,0.865,42,53
2,BM25,0.274,0.873,0.799,0.786,37,51
3,MMR,0.215,0.758,0.788,0.714,38,48
4,Metadata filtering,0.256,0.840,0.784,0.762,38,50
5,Parent-child,0.256,0.840,0.784,0.760,38,50
6,Sparse (SPLADE),0.281,0.862,0.783,0.785,36,49
7,Multi-query,0.233,0.752,0.761,0.701,37,46
8,TF-IDF,0.241,0.783,0.687,0.664,30,47
9,Contextual,0.215,0.705,0.661,0.644,31,42
10,Dense vector,0.215,0.675,0.610,0.599,26,41


### Which technique is good at which kind of question?

MRR for each question type (1.0 = the right passage always first):

In [12]:
by_type

,quote,paraphrase,names a book,topic
Hybrid,0.944,0.787,0.958,0.806
BM25,0.938,0.656,0.875,0.756
MMR,0.710,0.885,0.845,0.561
Metadata filtering,0.709,0.877,0.808,0.625
Parent-child,0.709,0.877,0.808,0.625
Sparse (SPLADE),0.787,0.833,0.669,0.833
Multi-query,0.654,0.887,0.713,0.724
TF-IDF,0.776,0.599,0.745,0.625
Contextual,0.513,0.833,0.667,0.469
Dense vector,0.444,0.713,0.667,0.597


### Can combining the best techniques do even better?

Hybrid merges two retrievers, so we can also try merging other strong ones:

In [13]:
from app.hybrid_retriever import HybridRetriever

parent_child, bm25 = techniques["Parent-child"], techniques["BM25"]
combinations = {
    "Hybrid (parent-child + BM25)": techniques["Hybrid"],
    "Multi-query + BM25": HybridRetriever([techniques["Multi-query"], bm25]),
    "Metadata filtering + BM25": HybridRetriever([techniques["Metadata filtering"], bm25]),
    "Hybrid + SPLADE": HybridRetriever([parent_child, bm25, techniques["Sparse (SPLADE)"]]),
}
compare_techniques(combinations, questions)[0]

,technique,precision@5,recall@5,MRR,nDCG@5,correct at #1,correct in top 5
1,Multi-query + BM25,0.300,0.952,0.883,0.875,43,54
2,Hybrid (parent-child + BM25),0.296,0.933,0.874,0.865,42,53
3,Metadata filtering + BM25,0.296,0.933,0.874,0.865,42,53
4,Hybrid + SPLADE,0.307,0.962,0.866,0.867,42,54


## Results

### The winner: Hybrid (parent-child dense search + BM25)

| # | Technique | Precision@5 | Recall@5 | MRR | nDCG@5 | Correct at #1 | In top 5 |
|---|---|---|---|---|---|---|---|
| 1 | **Hybrid** | **0.296** | **0.933** | **0.874** | **0.865** | **42** | **53** |
| 2 | BM25 | 0.274 | 0.873 | 0.799 | 0.786 | 37 | 51 |
| 3 | MMR | 0.215 | 0.758 | 0.788 | 0.714 | 38 | 48 |
| 4 | Metadata filtering | 0.256 | 0.840 | 0.784 | 0.762 | 38 | 50 |
| 5 | Parent-child | 0.256 | 0.840 | 0.784 | 0.760 | 38 | 50 |
| 6 | Sparse (SPLADE) | 0.281 | 0.862 | 0.783 | 0.785 | 36 | 49 |
| 7 | Multi-query | 0.233 | 0.752 | 0.761 | 0.701 | 37 | 46 |
| 8 | TF-IDF | 0.241 | 0.783 | 0.687 | 0.664 | 30 | 47 |
| 9 | Contextual | 0.215 | 0.705 | 0.661 | 0.644 | 31 | 42 |
| 10 | Dense vector | 0.215 | 0.675 | 0.610 | 0.599 | 26 | 41 |

Hybrid is first on **all four** metrics.

### Why Hybrid wins

**Its two halves are good at different things, so together they cover each other's mistakes.**

| | Quote | Paraphrase | Names a book | Topic |
|---|---|---|---|---|
| BM25 (exact words) | **0.938** | 0.656 | 0.875 | 0.756 |
| Parent-child (meaning) | 0.709 | **0.877** | 0.808 | 0.625 |
| **Hybrid (both)** | **0.944** | 0.787 | **0.958** | **0.806** |

- BM25 is excellent on quotes but struggles with paraphrases, because *"big fish"* shares no words with *"great fish"*.
- Dense search is the reverse: strong on paraphrases, weaker on exact quotes.
- Hybrid is first or close to first on every type, and finds a right passage in the top 5 for **53 of the 54** questions. For example, it puts John 3:16 first for *"For God so loved the world"*, which dense search alone doesn't even place in its top 5.

### Why not one of the combinations?

| Combination | Correct at #1 | In top 5 | MRR | Why we didn't pick it |
|---|---|---|---|---|
| Multi-query + BM25 | 43 | 54 | 0.883 | Ahead by only **one** question. Its KJV word list shares words with the test questions, so the lead may not hold on new questions. |
| **Hybrid** | **42** | **53** | **0.874** | **Chosen**: simple, fast, no hand-made word list |
| Metadata filtering + BM25 | 42 | 53 | 0.874 | Exactly the same scores as Hybrid, with extra code |
| Hybrid + SPLADE | 42 | 54 | 0.866 | Lower MRR, and every search runs a second neural model |

**Decision: use Hybrid** to find candidates. Step 8 tests whether a reranker can put them in a better order.

### What we learned about the other techniques

| Technique | What happened | Why |
|---|---|---|
| **Dense vector** (last) | Embedding whole 10-verse passages scores much lower than parent-child (0.610 vs 0.784) | The vector of a long passage is a blur of all its verses, so one specific verse gets lost. **Search small, return big.** |
| **Metadata filtering** | Exactly the same as parent-child | The filter works, but every chunk already starts with its reference (*"Luke 15:11-32 …"*), so dense search already favours the book a question names. The filter only changes lower ranks. |
| **Contextual** | Worse than parent-child (0.661 vs 0.784) | The added line is much the same for every chunk in a book, which blurs the differences between them. An LLM-written context, as in the original method, might help. |
| **MMR** | Slightly better MRR, but lower recall@5 (0.758 vs 0.840) | It pushes out the next part of the same story, which often holds the other answer verses. |
| **Multi-query** | Best on paraphrases (0.887), weaker on quotes | KJV rewording helps paraphrases, but the keyword-only version loses the exact wording quotes need. |
| **SPLADE** | Best on topics (0.833) | It adds related words a passage doesn't literally contain, which suits broad topic questions. |
| **TF-IDF** | Below BM25 on every type | BM25 improves on TF-IDF: repeated words have diminishing returns, and passage length is accounted for. |

### Try the winner

In [14]:
from app.techniques import build_hybrid

hybrid = build_hybrid(DATA_DIR)
for passage in hybrid.search("For God so loved the world", k=3):
    print(passage["text"][:200], "...\n")

John 3:15-21 (Jesus and Nicodemus)
15 That whosoever believeth in him should not perish, but have eternal life. 16 For God so loved the world, that he gave his only begotten Son, that whosoever believ ...

1 John 2:15-17 (Do Not Love the World)
15 Love not the world, neither the things [that are] in the world. If any man love the world, the love of the Father is not in him. 16 For all that [is] in the w ...

1 John 4:7-14 (Love Comes from God)
7 Beloved, let us love one another: for love is of God; and every one that loveth is born of God, and knoweth God. 8 He that loveth not knoweth not God; for God is  ...



## 8. Reranking: can a second model put Hybrid's results in a better order?

Hybrid puts a right passage in the top 5 for 53 of 54 questions, but **first** for only 42. The right passage is usually *there*, just not at the top. That is the job of a **reranker**.

### Two stages: fast search, then careful reading

1. **Hybrid** searches all 4,723 passages in a fraction of a second and keeps its **top 10**.
2. A **reranker** reads only those 10 passages more carefully, scores each one, and puts them in a new order.

A slow, careful model is affordable here because it reads 10 passages, not 4,723.

### Bi-encoder vs cross-encoder

| | **Bi-encoder** | **Cross-encoder** |
|---|---|---|
| How it reads | The question and passage **separately**: each becomes a vector, then the vectors are compared | The question and passage **together**, as one input, and outputs one relevance score |
| Strength | Fast, because passage vectors can be computed in advance | More accurate, because it sees how each word of the question relates to each word of the passage |
| Model tested | `multi-qa-mpnet-base-cos-v1`, trained on question → answer pairs | `cross-encoder/ms-marco-MiniLM-L-6-v2` (small, fast) and `BAAI/bge-reranker-base` (larger) |

All three run on this computer, with no API and no LLM. Code: `app/reranker.py`.

In [15]:
from app.reranker import build_rerankers

rerankers = build_rerankers(hybrid)  # Hybrid alone, then Hybrid + each reranker
rerank_overall, rerank_by_type = compare_techniques(rerankers, questions)
rerank_overall

,technique,precision@5,recall@5,MRR,nDCG@5,correct at #1,correct in top 5
1,Hybrid + cross-encoder (bge),0.307,0.963,0.945,0.930,49,54
2,Hybrid + cross-encoder (MiniLM),0.289,0.907,0.886,0.862,44,52
3,Hybrid (no reranker),0.296,0.933,0.874,0.865,42,53
4,Hybrid + bi-encoder,0.248,0.764,0.720,0.684,33,45


MRR for each question type:

In [16]:
rerank_by_type

,quote,paraphrase,names a book,topic
Hybrid + cross-encoder (bge),1.000,0.910,0.903,1.000
Hybrid + cross-encoder (MiniLM),0.882,0.820,1.000,0.889
Hybrid (no reranker),0.944,0.787,0.958,0.806
Hybrid + bi-encoder,0.607,0.774,0.743,0.792


### How long does each take?

Time per question on this CPU, averaged over 11 of the questions. Timings depend on what else the computer is doing: measured with nothing else running, bge took about 3.3 seconds and the bi-encoder about 4.1, which are the figures used in the results below.

In [17]:
import time

timings = {}
for name, retriever in rerankers.items():
    start = time.perf_counter()
    for q in questions[::5]:
        retriever.search(q["question"], 10)
    timings[name] = round((time.perf_counter() - start) / len(questions[::5]), 2)
pd.Series(timings, name="seconds per question")

Hybrid (no reranker)               0.14
Hybrid + bi-encoder                6.62
Hybrid + cross-encoder (MiniLM)    1.01
Hybrid + cross-encoder (bge)       6.14
Name: seconds per question, dtype: float64

## Reranking results

### The winner: the bge cross-encoder

| | Precision@5 | Recall@5 | MRR | nDCG@5 | Correct at #1 | In top 5 | Seconds per question |
|---|---|---|---|---|---|---|---|
| **Hybrid + cross-encoder (bge)** | **0.307** | **0.963** | **0.945** | **0.930** | **49** | **54** | ~3.3 |
| Hybrid + cross-encoder (MiniLM) | 0.289 | 0.907 | 0.886 | 0.862 | 44 | 52 | ~0.7 |
| Hybrid (no reranker) | 0.296 | 0.933 | 0.874 | 0.865 | 42 | 53 | ~0.05 |
| Hybrid + bi-encoder | 0.248 | 0.764 | 0.720 | 0.684 | 33 | 45 | ~4.1 |

- **bge improves all four metrics.** The right passage comes first for **49 of 54** questions (up from 42), and is in the top 5 for **all 54**.
- **Every quote and every topic question now has a right passage first** (MRR 1.000 on both).
- **Its precision@5 is 0.307**, close to the 0.337 that a perfect search would score.

### Why not the others?

| Reranker | What happened | Why |
|---|---|---|
| **MiniLM cross-encoder** | 2 more questions right at #1, but lower recall@5 and nDCG@5, and worse on quotes (0.882 vs 0.944) | A small model: it helps the top spot a little, but sometimes pushes a right passage out of the top 5 |
| **Bi-encoder** | Worse than no reranker on every metric; quotes fall from 0.944 to 0.607 | It throws away Hybrid's order, including BM25's exact word matching, and replaces it with one vector comparison, which is the weakest kind of search on this text (see Dense vector in step 7) |

### Making bge fast enough

bge is the slowest good option, so two settings were tested to speed it up:

| Setting | Effect on accuracy | Effect on time |
|---|---|---|
| Re-score the **top 10** instead of the top 20 | None: exactly the same scores, because every passage bge moved up was already in Hybrid's top 10 | About half |
| Score **one passage at a time** instead of in a batch | None | About 30% less, because a batch is padded to its longest passage, which wastes work on a CPU |

Together, these brought it from about 9 seconds per question to about 3.3. On a GPU it would take a fraction of a second.

**Decision: use Hybrid + bge reranking.** It is what `python -m app.main search` uses by default. Add `--technique hybrid` for a faster search without the reranker.

### Try the final search

In [18]:
from app.techniques import best_retriever

final = best_retriever(DATA_DIR)
for passage in final.search("Jonah is swallowed by a big fish", k=3):
    print(passage["text"][:200], "...\n")

Jonah 1:11-17 (Jonah Cast into the Sea)
11 Then said they unto him, What shall we do unto thee, that the sea may be calm unto us? for the sea wrought, and was tempestuous. 12 And he said unto them, Ta ...

Jonah 2:1-10 (Jonah’s Prayer)
1 Then Jonah prayed unto the LORD his God out of the fish’s belly, 2 And said, I cried by reason of mine affliction unto the LORD, and he heard me; out of the belly of he ...

Matthew 12:38-42 (The Sign of Jonah)
38 Then certain of the scribes and of the Pharisees answered, saying, Master, we would see a sign from thee. 39 But he answered and said unto them, An evil and adu ...



## What's next

Retrieval is finished. The next step is **generation**: hand the top passages to an LLM, which writes an answer and cites the verses it used.